# Oseen MHM: write momentum, incompressibility and interface forms

This notebook follows **meshes → local equations → global equations → assemble
→ solve → velocity and pressure**. A user writes mixed UFL forms directly;
PyMHM supplies mesh-associated trace numbering, orientation and condensation.

We first solve an independently manufactured affine problem exactly in stable
Taylor–Hood P2/P1 local spaces. Its polynomial exactness is a correctness check,
not a convergence rate. The final section reads the separately attributed smooth
stabilized P3/P3 refinement family and plots its measured asymptotic rates.

Run with the locked `introduction` profile. No optional native cell is silently
skipped.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/12939e9a0eb1240d14240ea26d81c4bf8e405ccd7ea2c48ec0727d9d001bdc87/oseen_variational-companion.zip"
COMPANION_SHA256 = "12939e9a0eb1240d14240ea26d81c4bf8e405ccd7ea2c48ec0727d9d001bdc87"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("flow/oseen_variational.ipynb", directory=ROOT)


In [ ]:
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("PYMHM_WORKSPACE", Path.cwd())).resolve()
if not (ROOT / "examples").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "examples").is_dir()), ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "build" / "native-teaching"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Workspace:", ROOT)


## 1. Prescribe the physical solution before defining a matrix

Take viscosity one, resistance one, and constant prescribed convection
$\beta=(2,3)$. The nonconservative Oseen equation is

$$
-\Delta u+(\beta\cdot\nabla)u+u+\nabla p=f,\qquad \nabla\cdot u=0.
$$

Choose

$$
u=(1-x+2y,-2-x+y),\quad p=x+2y-0.8,\quad f=u+(5,3).
$$

The gradient of velocity is constant, its divergence vanishes, and its
Laplacian is zero. Hence $(\beta\cdot\nabla)u=(4,1)$ and $\nabla p=(1,2)$.
The physical pressure mean is 0.7. Prescribe the exact velocity on every
exterior macroface; pressure is fixed by its global mean, not by one mean per
local problem.


In [ ]:
import numpy as np
import ufl
import basix
import basix.ufl
from dolfinx import fem
from pymhm import (
    TriangleMesh, FaceSpace, SkeletonSpace, Equation, MeshHierarchy,
    LocalContext, columns, compile_form, bind_interface, bind_problem, assemble, solve,
)

def exact_velocity(x):
    """Return the solenoidal affine physical velocity."""
    return np.column_stack((1-x[:,0]+2*x[:,1], -2-x[:,0]+x[:,1]))

def exact_pressure(x):
    """Return pressure with declared physical unit-square mean 0.7."""
    return x[:,0]+2*x[:,1]-0.8

macro=TriangleMesh.unit_square(2)
skeleton=SkeletonSpace(macro,tuple(FaceSpace.uniform(1) for _ in macro.faces),2)
hierarchy=MeshHierarchy(macro,tuple(macro.submesh(i,2) for i in range(len(macro.cells))))
interface=bind_interface(skeleton,convention="normal")


taylor_hood_element=basix.ufl.mixed_element([
    basix.ufl.element("Lagrange","triangle",2,shape=(2,),lagrange_variant=basix.LagrangeVariant.equispaced),
    basix.ufl.element("Lagrange","triangle",1,lagrange_variant=basix.LagrangeVariant.equispaced),
])


## 2. Write the local mixed energy and both interface pairings

Use the skew convection form

$$
\begin{aligned}
a_K((u,p),(v,q))={}&(\nabla u,\nabla v)_K+(u,v)_K\\
&+\tfrac12[(\beta\cdot\nabla u,v)_K-(u,\beta\cdot\nabla v)_K]\\
&-(p,\nabla\cdot v)_K-(q,\nabla\cdot u)_K.
\end{aligned}
$$

Since $\nabla\cdot\beta=0$, the effective reaction is one. The normal interface
quantity is the **half-advection pseudotraction**

$$
\lambda_K=(-\nabla u+pI+\tfrac12u\otimes\beta)n_K.
$$

The interface action is $+\langle\lambda_K,v\rangle$; the global velocity pairing
is $-\langle\mu_K,u\rangle$. `trace_pairings` supplies the physical supports and
`local.equations` applies the normal orientation once. No face permutation or
orientation code is written here.

Two translations are retained through physical velocity-integral moments.
They are **coarse modes**, not null vectors when drag is positive. The local
mixed pressure has no independently imposed mean. Named fields retain the
native executed coefficient basis for later evaluation.


In [ ]:
def oseen_volume_forms(W, domain):
    """Write the common physical UFL operator, source and pressure integral."""
    u,p=ufl.TrialFunctions(W)
    v,q=ufl.TestFunctions(W)
    x=ufl.SpatialCoordinate(domain)
    beta=ufl.as_vector((2.0,3.0))
    exact=ufl.as_vector((1-x[0]+2*x[1],-2-x[0]+x[1]))
    force=exact+ufl.as_vector((5.0,3.0))
    dx=ufl.Measure("dx",domain=domain,metadata={"quadrature_degree":8})
    a=(ufl.inner(ufl.grad(u),ufl.grad(v))+ufl.inner(u,v)
       +0.5*(ufl.inner(ufl.grad(u)*beta,v)-ufl.inner(u,ufl.grad(v)*beta))
       -p*ufl.div(v)-q*ufl.div(u))*dx
    L=ufl.inner(force,v)*dx
    return a,L,q*dx

def local_oseen(local: LocalContext):
    """Declare one Taylor–Hood Oseen problem in its executed native basis."""
    binding=local.native_space(taylor_hood_element)
    W=binding.space
    u,p=ufl.TrialFunctions(W)
    v,q=ufl.TestFunctions(W)
    a,L,mean_form=oseen_volume_forms(W,binding.mesh)
    dx=ufl.Measure("dx",domain=binding.mesh,metadata={"quadrature_degree":8})
    b=local.trace_pairings(lambda phi,ds:ufl.inner(phi,v)*ds)
    c=local.trace_pairings(lambda phi,ds:-ufl.inner(phi,u)*ds,axis="rows")
    _,velocity_dofs=W.sub(0).collapse()
    translations=np.zeros((binding.size,2))
    translations[velocity_dofs]=np.tile(np.eye(2),(len(velocity_dofs)//2,1))
    local.field("velocity",binding,component=0)
    local.field("pressure",binding,component=1)
    return local.equations(
        a=a,L=L,b=b,c=c,
        coarse_basis=translations,moments=columns(v[0]*dx,v[1]*dx),
        metadata={"pressure_weights":compile_form(q*dx)},
    )


## 3. Declare the global velocity boundary equation and pressure gauge

Local `c` equations sum signed velocity continuity and exterior velocity
moments. The additional global functional has the matching minus sign.
The global physical pressure constraint is imposed **after assembly**, because
its row depends on the reconstructed local pressure responses.


In [ ]:
def global_equation(global_context):
    """Insert the prescribed velocity moments in the declared global convention."""
    boundary,_=global_context.boundary_data(exact_velocity,order=6)
    return Equation(0,-global_context.trace_load(boundary))

problem=bind_problem(hierarchy,interface,local_oseen,
                     global_equation=global_equation,retained=2)
system=assemble(problem)
pressure_weights=[record["pressure_weights"] for record in system.local_metadata]
gauge=system.mean_constraint(pressure_weights,0.7)
solution=solve(system,constraints=(gauge,))
velocity=solution.field("velocity")
pressure=solution.field("pressure")


## 4. Measure independent physical fields and original equations

The exact affine functions belong to the declared local spaces. Check both
fields at independent random physical points, and integrate the pressure mean
through its assembled physical weights. Zero errors in a polynomial patch
are roundoff controls and are not fitted as convergence slopes.


In [ ]:
maximum={"velocity":0.0,"pressure":0.0}
for vf,pf in zip(velocity,pressure,strict=True):
    bary=np.random.default_rng(127).dirichlet([1,1,1],size=40)
    vertices=vf.mesh.points[vf.mesh.cells[0]]
    points=bary@vertices
    maximum["velocity"]=max(maximum["velocity"],float(np.max(abs(vf.evaluate(points)-exact_velocity(points)))))
    maximum["pressure"]=max(maximum["pressure"],float(np.max(abs(pf.evaluate(points)-exact_pressure(points)))))
mean=sum(float(w@values) for w,values in zip(pressure_weights,solution.fields,strict=True))
assert max(maximum.values())<1e-10
np.testing.assert_allclose(mean,0.7,rtol=1e-10,atol=1e-12)
print({"maximum_physical_errors":maximum,"pressure_mean":mean,
       "global_compatibility_relative":solution.raw_residual})


## 5. Independently assemble a fine classical Taylor–Hood reference

The same written UFL volume operator is assembled on one globally conforming
P2/P1 space, without local condensation. Strong exterior velocity and the same
physical pressure mean make the reference a matching physical problem. Two
fine meshes independently reproduce this affine solution; errors near roundoff
are exactness controls and do not define a fitted rate.


In [ ]:
from pymhm import MultiscaleProblem
from pymhm.backends.spaces import bind_space
from pymhm.postprocessing.fields import DiscreteField,FieldDefinition

def classical_oseen(n):
    """Assemble the same mixed forms globally with strong velocity and one pressure mean."""
    fine=TriangleMesh.unit_square(n)
    binding=bind_space(fine,taylor_hood_element)
    try:
        W=binding.space
        a,L,pressure_mean=oseen_volume_forms(W,binding.mesh)
        A,F,m=compile_form(a),compile_form(L),compile_form(pressure_mean)
        V,vmap=W.sub(0).collapse()
        coordinates=V.tabulate_dof_coordinates()[:,:2]
        boundary=np.flatnonzero(np.any(np.isclose(coordinates,0)|np.isclose(coordinates,1),axis=1))
        fixed_dofs=np.asarray(vmap)[(2*boundary[:,None]+np.arange(2)).ravel()]
        values=exact_velocity(coordinates[boundary]).ravel()
        definition=MultiscaleProblem.from_global(Equation(A,F),binding.size,
            fixed=dict(zip(fixed_dofs.tolist(),values.tolist(),strict=True)),constraints=((m,0.7),))
        reference=solve(assemble(definition))
        velocity=DiscreteField(FieldDefinition("velocity",descriptor=binding.descriptor(component=0)),reference.trace)
        pressure=DiscreteField(FieldDefinition("pressure",descriptor=binding.descriptor(component=1)),reference.trace)
        points=np.random.default_rng(137).uniform(0,1,size=(100,2))
        checks={"n":n,"velocity_max":float(np.max(abs(velocity.evaluate(points)-exact_velocity(points)))),
                "pressure_max":float(np.max(abs(pressure.evaluate(points)-exact_pressure(points)))),
                "pressure_integral":float(m@reference.trace)}
        assert max(checks["velocity_max"],checks["pressure_max"])<1e-9
        np.testing.assert_allclose(checks["pressure_integral"],0.7,rtol=1e-10,atol=1e-12)
        return velocity,pressure,checks
    finally:
        binding.close()

reference_rows=[]
for n in (8,16):
    reference_velocity,reference_pressure,row=classical_oseen(n)
    reference_rows.append(row)
print("Independent conforming Taylor–Hood controls:",reference_rows)


## 5. Plot velocity and pressure

Every panel displays the actual macro mesh. Macrocells are plotted independently,
retaining their own incident field values. The analytical solution is the exact
accuracy reference for this polynomial control.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

def scalar_panel(ax, fields, exact, title, *, error=False):
    """Plot each macrocell independently, with original macro topology visible."""
    artists=[]
    for field in fields:
        fine=field.mesh
        values=field.evaluate(fine.points)
        if values.ndim==2: values=np.linalg.norm(values,axis=1)
        if exact is not None:
            truth=exact(fine.points)
            if truth.ndim==2: truth=np.linalg.norm(truth,axis=1)
            values=abs(values-truth) if error else truth
        artist=ax.tripcolor(fine.points[:,0],fine.points[:,1],fine.cells,values,shading="gouraud")
        artists.append(artist)
    lower=min(a.get_clim()[0] for a in artists);upper=max(a.get_clim()[1] for a in artists)
    for a in artists:a.set_clim(lower,upper)
    ax.add_collection(LineCollection(macro.points[macro.faces],colors="black",linewidths=0.65,alpha=0.55))
    ax.set(title=title,xlabel="x",ylabel="y",aspect="equal")
    return artists[-1]
figure,axes=plt.subplots(2,4,figsize=(15,7),layout="constrained")
for row,fields,exact,name in [(0,velocity,exact_velocity,"Velocity magnitude"),(1,pressure,exact_pressure,"Pressure")]:
    for column,title,truth,error in [(0,"Analytical",exact,False),(1,"MHM",None,False),(2,"Classical Taylor–Hood",None,False),(3,"Absolute error",exact,True)]:
        chosen=(reference_velocity,) if row==0 and column==2 else (reference_pressure,) if row==1 and column==2 else fields
        artist=scalar_panel(axes[row,column],chosen,truth,f"{title}\n{name.lower()}",error=error)
        figure.colorbar(artist,ax=axes[row,column],pad=0.025,shrink=0.85)
figure.savefig(OUTPUT/"oseen-affine-fields.png",dpi=180)
plt.show()


## 6. Define the nonconstant smooth convergence problem

Now write the same smooth problem used in the refinement curves. The stream
function guarantees divergence-free velocity, and pressure has integral zero:

$$
\begin{aligned}
\psi&=-128x^2(1-x)^2y^2(1-y)^2,\\
u_*&=(\partial_y\psi,-\partial_x\psi),
&p_*&=(x-y)^6-1/28,\\
f&=-\Delta u_*+(\nabla u_*)\beta+u_*+\nabla p_*,
&\beta&=(1,1)/\sqrt2.
\end{aligned}
$$

Velocity is zero on the exterior. The local equal-order P3/P3 fields require
the consistent Oseen stabilization; Taylor–Hood is a different stable local
choice. Write both the trial and adjoint-test residual before assembling:

$$
\begin{aligned}
R(u,p)&=-\Delta u+(\nabla u)\beta+u+\nabla p,\\
R^*(v,q)&=-\Delta v-(\nabla v)\beta+v+\nabla q,\\
a_K^{\rm US}&=a_K+\kappa_K(\nabla\cdot u,\nabla\cdot v)_K
                 -\delta_K(R(u,p),R^*(v,q))_K,\\
L_K^{\rm US}&=(f,v)_K-\delta_K(f,R^*(v,q))_K.
\end{aligned}
$$

The minus sign and stabilized source are part of the formulation. For the
computed polynomial inverse constant $m_K$, set $d_K=4/m_K$, $b_K=h_K$,
$\delta_K=h_K^2/[\max(h_K^2,d_K)+\max(d_K,b_K)]$ and
$\kappa_K=b_K\min(1,b_K/d_K)$. The generic inverse-bound routine uses Basix
polynomial derivatives; it does not choose the physical equation for us.


In [ ]:
from dolfinx import default_scalar_type
from pymhm.fem.inequalities import laplacian_inverse_bound
from pymhm.fem.scalar.operators import triangle_quadrature
from pymhm.fem.scalar.triangle import tabulate
from examples.solve_oseen import crisscross


def smooth_velocity(points):
    """Differentiate the analytical stream function independently of assembly."""
    x,y=points.T
    sx,sy=x*x*(1-x)**2,y*y*(1-y)**2
    dsx,dsy=2*x*(1-x)*(1-2*x),2*y*(1-y)*(1-2*y)
    return np.column_stack((-128*sx*dsy,128*dsx*sy))


def smooth_pressure(points):
    """Return the independently stated pressure with zero physical mean."""
    return (points[:,0]-points[:,1])**6-1/28


def smooth_forms(W,domain,*,delta=0.0,kappa=0.0):
    """Declare the same physical volume equation and optional consistent residuals."""
    u,p=ufl.TrialFunctions(W)
    v,q=ufl.TestFunctions(W)
    x=ufl.SpatialCoordinate(domain)
    beta=ufl.as_vector((1/np.sqrt(2),1/np.sqrt(2)))
    psi=-128*x[0]**2*(1-x[0])**2*x[1]**2*(1-x[1])**2
    truth=ufl.as_vector((psi.dx(1),-psi.dx(0)))
    truth_p=(x[0]-x[1])**6-1/28
    force=-ufl.div(ufl.grad(truth))+ufl.grad(truth)*beta+truth+ufl.grad(truth_p)
    dx=ufl.Measure('dx',domain=domain,metadata={'quadrature_degree':20})
    physical=(ufl.inner(ufl.grad(u),ufl.grad(v))+ufl.inner(u,v)
              +0.5*(ufl.inner(ufl.grad(u)*beta,v)-ufl.inner(u,ufl.grad(v)*beta))
              -p*ufl.div(v)-q*ufl.div(u))*dx
    residual=-ufl.div(ufl.grad(u))+ufl.grad(u)*beta+u+ufl.grad(p)
    adjoint=-ufl.div(ufl.grad(v))-ufl.grad(v)*beta+v+ufl.grad(q)
    a=physical+(kappa*ufl.div(u)*ufl.div(v)-delta*ufl.inner(residual,adjoint))*dx
    L=(ufl.inner(force,v)-delta*ufl.inner(force,adjoint))*dx
    return a,L,q*dx


equal_order=basix.ufl.mixed_element([
    basix.ufl.element('Lagrange','triangle',3,shape=(2,),
                      lagrange_variant=basix.LagrangeVariant.equispaced),
    basix.ufl.element('Lagrange','triangle',3,
                      lagrange_variant=basix.LagrangeVariant.equispaced),
])


def smooth_local_equations(local):
    """Write enriched local momentum, incompressibility and signed face pairings."""
    fine=local.mesh
    bary,weights=triangle_quadrature(10)
    _,_,_,gradient,hessian=tabulate(fine,3,bary)
    h=np.max(fine.lengths[fine.cell_faces],axis=1)
    inverse=laplacian_inverse_bound(gradient,hessian,weights,h)
    d=4/inverse
    delta=float((h*h/(np.maximum(h*h,d)+np.maximum(d,h)))[0])
    kappa=float((h*np.minimum(1,h/d))[0])
    binding=local.native_space(equal_order)
    W=binding.space
    u,p=ufl.TrialFunctions(W)
    v,q=ufl.TestFunctions(W)
    delta_coefficient=fem.Constant(binding.mesh,default_scalar_type(delta))
    kappa_coefficient=fem.Constant(binding.mesh,default_scalar_type(kappa))
    a,L,pressure_mean=smooth_forms(W,binding.mesh,delta=delta_coefficient,kappa=kappa_coefficient)
    dx=ufl.Measure('dx',domain=binding.mesh,metadata={'quadrature_degree':20})
    b=local.trace_pairings(lambda phi,ds:ufl.inner(phi,v)*ds)
    c=local.trace_pairings(lambda phi,ds:-ufl.inner(phi,u)*ds,axis='rows')
    _,velocity_rows=W.sub(0).collapse()
    translations=np.zeros((binding.size,2))
    translations[velocity_rows]=np.tile(np.eye(2),(len(velocity_rows)//2,1))
    local.field('velocity',binding,component=0)
    local.field('pressure',binding,component=1)
    return local.equations(
        a=a,L=L,b=b,c=c,coarse_basis=translations,
        moments=columns(v[0]*dx,v[1]*dx),
        metadata={'pressure_weights':compile_form(pressure_mean)},
    )


smooth_macro=crisscross(8)
smooth_trace=SkeletonSpace(
    smooth_macro,tuple(FaceSpace.uniform(1) for _ in smooth_macro.faces),components=2,
)
smooth_hierarchy=MeshHierarchy(
    smooth_macro,tuple(smooth_macro.submesh(i,1) for i in range(len(smooth_macro.cells))),
)
smooth_problem=bind_problem(
    smooth_hierarchy,bind_interface(smooth_trace,convention='normal'),
    smooth_local_equations,global_equation=Equation(0,0),retained=2,
)
smooth_system=assemble(smooth_problem)
smooth_weights=[item['pressure_weights'] for item in smooth_system.local_metadata]
smooth_gauge=smooth_system.mean_constraint(smooth_weights,0.0)
smooth_solution=solve(smooth_system,constraints=(smooth_gauge,))
smooth_u,smooth_p=smooth_solution.field('velocity'),smooth_solution.field('pressure')


## 7. Check the physical norms and independently refine Taylor–Hood

The product norm used by Araya et al. (2021), Section 2.2, is

$$
\lVert(e_u,e_p)\rVert_{V\times Q}^2
=\tfrac12\lVert e_u\rVert_{L^2}^2
 +\lVert\nabla e_u\rVert_{L^2(\mathcal T_H)}^2
 +\lVert e_p\rVert_{L^2}^2.
$$

Its smooth P1-trace target is order two. Velocity L2 order three is separately
observed additional accuracy. Integrate the physical fields at independent
Gaussian points, keeping the incident local cell identifiers explicit. The
original-row observer checks the complete executed mixed blocks and pressure
gauge separately; an algebraic residual is not this physical product norm.

For the classical baseline, use the same `smooth_forms` with delta and kappa
zero in the stable global P2/P1 Taylor–Hood space. Strong zero velocity and
one zero pressure integral match the physical problem. Its refinement errors
must decrease before using the finest numerical field as a baseline.


In [ ]:
import json
from types import SimpleNamespace
from examples.core_elasticity_field_archive import ProductionObservation
from examples.minimal_flow_originals import original_diagnostics
from pymhm.fem.vector.curl import quadrature,physical_points


def smooth_errors(velocity_fields,pressure_fields,order):
    """Integrate physical velocity, pressure and broken-gradient errors independently."""
    sums=np.zeros(3,dtype=np.longdouble)
    for vf,pf in zip(velocity_fields,pressure_fields,strict=True):
        fine=vf.mesh
        bary,weights,_=quadrature(fine,1,order)
        points=physical_points(fine,bary)
        cells=np.repeat(np.arange(len(fine.cells)),points.shape[1])
        flat=points.reshape(-1,2)
        numerical_u,numerical_gradient=vf.values_and_gradient(flat,cells=cells)
        numerical_p=pf.evaluate(flat,cells=cells)
        x,y=flat.T
        sx,sy=x*x*(1-x)**2,y*y*(1-y)**2
        dx,dy=2*x*(1-x)*(1-2*x),2*y*(1-y)*(1-2*y)
        ddx,ddy=2-12*x+12*x*x,2-12*y+12*y*y
        exact_gradient=-128*np.array([[dx*dy,sx*ddy],[-ddx*sy,-dx*dy]]).transpose(2,0,1)
        eu=numerical_u-smooth_velocity(flat)
        ep=numerical_p-smooth_pressure(flat)
        eg=numerical_gradient-exact_gradient
        w=weights.ravel()
        sums+=np.array([np.sum(w*np.sum(eu*eu,axis=1),dtype=np.longdouble),
                        np.sum(w*ep*ep,dtype=np.longdouble),
                        np.sum(w*np.sum(eg*eg,axis=(1,2)),dtype=np.longdouble)])
    return {'velocity_l2':float(np.sqrt(sums[0])),
            'pressure_l2':float(np.sqrt(sums[1])),
            'mixed_error':float(np.sqrt(sums[0]/2+sums[1]+sums[2]))}


observation=ProductionObservation(
    system=smooth_system,applied_boundary=np.zeros(smooth_system.trace_size),
    mean_weights=[tuple(smooth_weights)],mean_values=[0.0],constraints=[smooth_gauge],
)
smooth_original=original_diagnostics(
    SimpleNamespace(hybrid=smooth_solution),observation,
    [{'mixed_velocity_pressure':len(values)} for values in smooth_solution.fields],
)
assert smooth_original['accepted']
norms={order:smooth_errors(smooth_u,smooth_p,order) for order in (10,14)}
for name in norms[14]:
    np.testing.assert_allclose(norms[10][name],norms[14][name],rtol=1e-8,atol=1e-12)
record_path=ROOT/'examples/results/oseen/smooth-nu1-l1-uniform.json'
record=json.loads(record_path.read_text())
archived=next(row for row in record['rows'] if row['resolution']==8)
for name in norms[14]:
    np.testing.assert_allclose(norms[14][name],archived[name],rtol=1e-8,atol=1e-12)
print('Current by-hand UFL and archived smooth norms:',norms[14])
print('Original mixed rows:',{k:v for k,v in smooth_original.items() if k!='blocks'})


def classical_smooth_oseen(n):
    """Assemble the declared physical UFL operator globally in stable Taylor–Hood."""
    fine=TriangleMesh.unit_square(n)
    binding=bind_space(fine,taylor_hood_element)
    try:
        W=binding.space
        a,L,mean=smooth_forms(W,binding.mesh)
        A,F,m=compile_form(a),compile_form(L),compile_form(mean)
        V,parent=W.sub(0).collapse()
        points=V.tabulate_dof_coordinates()[:,:2]
        exterior=np.flatnonzero(np.any(np.isclose(points,0)|np.isclose(points,1),axis=1))
        fixed=np.asarray(parent)[(2*exterior[:,None]+np.arange(2)).ravel()]
        definition=MultiscaleProblem.from_global(
            Equation(A,F),binding.size,fixed={int(i):0.0 for i in fixed},constraints=((m,0.0),),
        )
        state=solve(assemble(definition))
        velocity=DiscreteField(FieldDefinition('velocity',descriptor=binding.descriptor(component=0)),state.trace)
        pressure=DiscreteField(FieldDefinition('pressure',descriptor=binding.descriptor(component=1)),state.trace)
        row={'n':n,**smooth_errors((velocity,),(pressure,),14),
             'global_compatibility_relative':state.raw_residual,'pressure_integral':float(m@state.trace)}
        np.testing.assert_allclose(row['pressure_integral'],0.0,atol=1e-10)
        return velocity,pressure,row
    finally:
        binding.close()


smooth_classical=[]
for n in (16,32,64):
    smooth_reference_u,smooth_reference_p,row=classical_smooth_oseen(n)
    smooth_classical.append(row)
assert all(smooth_classical[i+1][name]<smooth_classical[i][name]
           for i in range(2) for name in ('velocity_l2','pressure_l2','mixed_error'))
print('Independent conforming Taylor–Hood refinement:',smooth_classical)


In [ ]:
from examples.introduction.vector import plot_field_panels

display_triangle=TriangleMesh(((0,0),(1,0),(0,1)),((0,1,2),)).submesh(0,8)
reference_points=np.column_stack((1-display_triangle.points.sum(axis=1),display_triangle.points))
reference_triangles=display_triangle.cells
plot_points=[];plot_triangles=[];plot_u=[];plot_p=[];offset=0
for vf,pf in zip(smooth_u,smooth_p,strict=True):
    fine=vf.mesh
    local_points=reference_points@fine.points[fine.cells[0]]
    plot_points.append(local_points)
    plot_triangles.append(reference_triangles+offset)
    plot_u.append(vf.evaluate(local_points));plot_p.append(pf.evaluate(local_points))
    offset+=len(local_points)
plot_points=np.concatenate(plot_points);plot_triangles=np.concatenate(plot_triangles)
plot_u=np.concatenate(plot_u);plot_p=np.concatenate(plot_p)
truth_u=smooth_velocity(plot_points);truth_p=smooth_pressure(plot_points)
reference_u=smooth_reference_u.evaluate(plot_points)
reference_p=smooth_reference_p.evaluate(plot_points)
figure=plot_field_panels(smooth_macro,{
    'Analytical velocity magnitude':(plot_points,plot_triangles,np.linalg.norm(truth_u,axis=1)),
    'MHM velocity magnitude':(plot_points,plot_triangles,np.linalg.norm(plot_u,axis=1)),
    'Fine Taylor–Hood velocity magnitude':(plot_points,plot_triangles,np.linalg.norm(reference_u,axis=1)),
    'Analytical pressure':(plot_points,plot_triangles,truth_p),
    'MHM pressure':(plot_points,plot_triangles,plot_p),
    'Fine Taylor–Hood pressure':(plot_points,plot_triangles,reference_p),
    'Velocity error magnitude':(plot_points,plot_triangles,np.linalg.norm(plot_u-truth_u,axis=1)),
    'Signed pressure error':(plot_points,plot_triangles,plot_p-truth_p),
},figsize=(13,11))
figure.savefig(OUTPUT/'oseen-smooth-fields.png',dpi=200,
               metadata={'Author':'IPES Research Group','License':'CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/'})
plt.show()


## 8. End with the same smooth stabilized refinement family

The by-hand UFL definition above matches the n8 archived velocity, pressure and
product-norm errors. All levels preserve P3/P3 local fields on one triangle,
P1 vector traces and the crisscross geometry. The smooth product-norm target is
two, as stated in Araya et al. (2021), Sections 2.2 and 5.1. Velocity L2 order
three is labeled additional observed accuracy. Display every level, three
successive terminal rates and normalized amplitudes; do not infer a spatial
rate from the affine reproduction patch.


In [ ]:
import json
import hashlib
from examples.tutorial_convergence import plot_method_series
record_path=ROOT/"examples/results/oseen/smooth-nu1-l1-uniform.json"
record=json.loads(record_path.read_text())
rows=record["rows"]
sizes=np.array([row["H"] for row in rows])
errors={"mixed V x Q norm":[row["mixed_error"] for row in rows],"velocity L2":[row["velocity_l2"] for row in rows],"pressure L2":[row["pressure_l2"] for row in rows]}
rates={name:(np.log(np.array(values[:-1])/values[1:])/np.log(sizes[:-1]/sizes[1:])).tolist() for name,values in errors.items()}
series={"method":"oseen","sizes":sizes.tolist(),"errors":errors,"rates":rates,
        "expected":{"mixed V x Q norm":2.0,"pressure L2":2.0},"refinement":"Macro diameter H"}
print({"source_sha256":hashlib.sha256(record_path.read_bytes()).hexdigest(),"rates":rates})
figure=plot_method_series(series)
figure.savefig(OUTPUT/"oseen-convergence.png",dpi=180)
plt.show()


## References

- Araya, Cárcamo, Poza and Valentin (2021), *An adaptive multiscale hybrid-mixed
  method for the Oseen equations*. https://doi.org/10.1007/s10444-020-09833-8
- The smooth refinement uses the explicitly identified PyMHM mesh and spaces;
  it does not reconstruct unidentified historical adaptive meshes.
